# Build the final CCM result-grid table

This notebook bridges the per-dyad CCM runs (`2_CCM_single_dyad`) and the
multi-dyad summary figures in this section. Each dyad's CCM object grid
already stores, per (E, τ) configuration, the metrics the paper's summary statistics
need: Δρ (the color in Fig. 3/S6), final ρ at the optimal lag (Fig. S9), the optimal
lag itself (Fig. S7), and the surrogate-outperformance fractions that decide
significance (the half-moon annotations; Text S1.3). This notebook just gathers
those already-computed values across all ten dyad × treatment combinations (four
main-text dyads plus the six SI robustness dyads, each in untreated /
linearly-detrended / 1-kyr-high-pass form) into one tidy table, so the plotting
notebooks never have to touch individual CCM output files.

Think of it as a one-time export notebook: it converts the metrics already stored in
each upgraded CCM object grid into one lightweight, plot-ready parquet file,

`hol_temp_tsi_ccm/mixed/tmp/final_result_grid.parquet`

with one row per dyad, `(E, tau)`, and preprocessing category. It's built to feed the
multiplot result grids without loading raw data or the individual CCM parquet
outputs. The exporter deliberately reuses the metrics already saved in the upgraded
object grids rather than recalculating lags, so `selection_policy` honestly records
`object_grid_precomputed`.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from datetime import datetime, timezone
import json
import os

import pandas as pd

from cedarkit.utils.io.cloudjoblib import joblib_cloud_load

NOTEBOOK_NAME = "calc__final_result_grid"
SELECTION_POLICY = "object_grid_precomputed"


stem = Path(*(p := Path.cwd().resolve()).parts[: p.parts.index("notebooks")])
output_dir = stem / "hol_temp_tsi_ccm" / "mixed" / "tmp"
output_path = output_dir / "final_result_grid.parquet"
readme_path = output_dir / "final_result_grid_README.md"
print(f"Will write: {output_path}")


Will write: /Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm/mixed/tmp/final_result_grid.parquet


## Source-grid inventory

Edit only this cell when the project roster changes.  Each entry records the
dyad name and its figure row category.  The local Paleobook is tried first;
the workstation result roots are fallbacks so this exporter can be run before
the complete raw CCM output tree is copied into the book.


In [2]:
source_root_candidates = {
    "hol_temp_tsi_ccm": [
        stem / "hol_temp_tsi_ccm",
        Path("/Users/jlanders/PycharmProjects/CCM_software/hol_temp_tsi_ccm"),
    ],
    "hol_temp_tsi_ccm1k": [
        stem / "hol_temp_tsi_ccm1k",
        Path("/Users/jlanders/PycharmProjects/CCM_software/hol_temp_tsi_ccm1k"),
    ],
}

project_specs = [
    # label, category, project root, dyad name
    ("Tian-Wu", "original", "hol_temp_tsi_ccm", "Tian22HT115kaALLGMST_Wu18TSI"),
    ("Erb-Wu", "original", "hol_temp_tsi_ccm", "Erb22daGMST_Wu18TSI"),
    ("Erb-Vieira", "original", "hol_temp_tsi_ccm", "Erb22daGMST_Vieira11TSI"),
    ("Alley-Wu", "original", "hol_temp_tsi_ccm", "GISP2Alley00Tanom_Wu18TSI"),
    ("Alley-Vieira", "original", "hol_temp_tsi_ccm", "GISP2Alley00Tanom_Vieira11TSI"),
    ("Doering-Wu", "original", "hol_temp_tsi_ccm", "GISP2Doering22T15N_Wu18TSI"),
    ("GISP2 Seierstad-Wu", "original", "hol_temp_tsi_ccm", "GISP2Seierstad14d18O_Wu18TSI"),
    ("NGRIP Seierstad-Wu", "original", "hol_temp_tsi_ccm", "NGRIP1Seierstad14d18O_Wu18TSI"),
    ("GISP2 Martin-Wu", "original", "hol_temp_tsi_ccm", "GISP2Martin24Tanom_Wu18TSI"),
    ("NGRIP Martin-Wu", "original", "hol_temp_tsi_ccm", "NGRIPMartin24Tanom_Wu18TSI"),
    ("Tian-Wu", "lineardetrended", "hol_temp_tsi_ccm", "Tian22HT115kaALLGMSTLinear_Wu18TSILinear"),
    ("Erb-Wu", "lineardetrended", "hol_temp_tsi_ccm", "Erb22daGMSTLinear_Wu18TSILinear"),
    ("Erb-Vieira", "lineardetrended", "hol_temp_tsi_ccm", "Erb22daGMSTLinear_Vieira11TSILinear"),
    ("Alley-Wu", "lineardetrended", "hol_temp_tsi_ccm", "GISP2Alley00TanomLinear_Wu18TSILinear"),
    ("Alley-Vieira", "lineardetrended", "hol_temp_tsi_ccm", "GISP2Alley00TanomLinear_Vieira11TSILinear"),
    ("Doering-Wu", "lineardetrended", "hol_temp_tsi_ccm", "GISP2Doering22T15NLinear_Wu18TSILinear"),
    ("GISP2 Seierstad-Wu", "lineardetrended", "hol_temp_tsi_ccm", "GISP2Seierstad14d18OLinear_Wu18TSILinear"),
    ("NGRIP Seierstad-Wu", "lineardetrended", "hol_temp_tsi_ccm", "NGRIP1Seierstad14d18OLinear_Wu18TSILinear"),
    ("GISP2 Martin-Wu", "lineardetrended", "hol_temp_tsi_ccm", "GISP2Martin24TanomLinear_Wu18TSILinear"),
    ("NGRIP Martin-Wu", "lineardetrended", "hol_temp_tsi_ccm", "NGRIPMartin24TanomLinear_Wu18TSILinear"),
    ("Tian-Wu", "1kyrfir", "hol_temp_tsi_ccm1k", "tian22ht115kaallgmst1kyrfir_wu18tsianom1kyrfir"),
    ("Erb-Wu", "1kyrfir", "hol_temp_tsi_ccm1k", "erb22dagmst1kyrfir_wu18tsianom1kyrfir"),
    ("Erb-Vieira", "1kyrfir", "hol_temp_tsi_ccm1k", "erb22dagmst1kyrfir_vieira11tsianom1kyrfir"),
    ("Alley-Wu", "1kyrfir", "hol_temp_tsi_ccm1k", "alley00gisp2multiproxy1kyrfir_wu18tsianom1kyrfir"),
    ("Alley-Vieira", "1kyrfir", "hol_temp_tsi_ccm1k", "alley00gisp2multiproxy1kyrfir_vieira11tsianom1kyrfir"),
    ("Doering-Wu", "1kyrfir", "hol_temp_tsi_ccm1k", "doering22gisp2t15n1kyrfir_wu18tsianom1kyrfir"),
    ("GISP2 Seierstad-Wu", "1kyrfir", "hol_temp_tsi_ccm1k", "seierstad14gisp2d18o1kyrfir_wu18tsianom1kyrfir"),
    ("NGRIP Seierstad-Wu", "1kyrfir", "hol_temp_tsi_ccm1k", "seierstad14ngrip1d18o1kyrfir_wu18tsianom1kyrfir"),
    ("GISP2 Martin-Wu", "1kyrfir", "hol_temp_tsi_ccm1k", "martin24gisp2tanom1kyrfir_wu18tsianom1kyrfir"),
    ("NGRIP Martin-Wu", "1kyrfir", "hol_temp_tsi_ccm1k", "martin24ngriptanom1kyrfir_wu18tsianom1kyrfir"),
]


def find_grid_path(project_root, proj_name):
    for root in source_root_candidates[project_root]:
        candidate = root / proj_name / "tmp" / f"{proj_name}_obj_grid__update.joblib"
        if candidate.is_file():
            return candidate
    return None


inventory = pd.DataFrame([
    {"pair_label": label, "row_category": category, "project_root": root,
     "proj_name": name, "grid_path": str(find_grid_path(root, name) or "")}
    for label, category, root, name in project_specs
])
display(inventory)
assert inventory.grid_path.ne("").all(), "Missing upgraded object grids; fix the source roots before exporting."
assert inventory.proj_name.is_unique, "The export roster contains a duplicated dyad."


,pair_label,row_category,project_root,proj_name,grid_path
0,Tian-Wu,original,hol_temp_tsi_ccm,Tian22HT115kaALLGMST_Wu18TSI,/Users/jlanders/PycharmProjects/CCM_software/h...
1,Erb-Wu,original,hol_temp_tsi_ccm,Erb22daGMST_Wu18TSI,/Users/jlanders/PycharmProjects/CCM_software/h...
2,Erb-Vieira,original,hol_temp_tsi_ccm,Erb22daGMST_Vieira11TSI,/Users/jlanders/PycharmProjects/CCM_software/h...
3,Alley-Wu,original,hol_temp_tsi_ccm,GISP2Alley00Tanom_Wu18TSI,/Users/jlanders/PycharmProjects/CCM_software/h...
4,Alley-Vieira,original,hol_temp_tsi_ccm,GISP2Alley00Tanom_Vieira11TSI,/Users/jlanders/PycharmProjects/holocene_TSI_t...
5,Doering-Wu,original,hol_temp_tsi_ccm,GISP2Doering22T15N_Wu18TSI,/Users/jlanders/PycharmProjects/CCM_software/h...
6,GISP2 Seierstad-Wu,original,hol_temp_tsi_ccm,GISP2Seierstad14d18O_Wu18TSI,/Users/jlanders/PycharmProjects/CCM_software/h...
7,NGRIP Seierstad-Wu,original,hol_temp_tsi_ccm,NGRIP1Seierstad14d18O_Wu18TSI,/Users/jlanders/PycharmProjects/CCM_software/h...
8,GISP2 Martin-Wu,original,hol_temp_tsi_ccm,GISP2Martin24Tanom_Wu18TSI,/Users/jlanders/PycharmProjects/CCM_software/h...
9,NGRIP Martin-Wu,original,hol_temp_tsi_ccm,NGRIPMartin24Tanom_Wu18TSI,/Users/jlanders/PycharmProjects/CCM_software/h...


## Extract the stored selected metrics

This cell reads the object-grid metadata only.  It upgrades the legacy
`GridCell.output` storage convention in memory when necessary, extracts the
plot-relevant primitive values from `r1`, and validates that each dyad has a
complete `(E, tau)` grid.  It never follows the object's output paths.


In [3]:
METRIC_COLUMNS = [
    "delta_rho", "maxlibsize_rho", "lag", "peak_start", "peak_end",
    "surr_rx_count", "surr_rx_count_outperforming", "surr_rx_outperforming_frac",
    "surr_ry_count", "surr_ry_count_outperforming", "surr_ry_outperforming_frac",
]


def upgrade_legacy_grid_cell(cell_obj):
    if cell_obj is not None and not hasattr(cell_obj, "_outputs"):
        legacy_output = cell_obj.__dict__.pop("output", None)
        cell_obj._outputs = [] if legacy_output is None else [legacy_output]
    return cell_obj


def primitive(value):
    # Keep parquet-friendly scalar values; retain missing values as None.
    return value if isinstance(value, (str, int, float, bool)) or value is None else str(value)


records = []
problems = []
metric_gaps = []
for spec in inventory.itertuples(index=False):
    object_grid = joblib_cloud_load(spec.grid_path)
    dyad_records = []
    for _, cell_obj in object_grid.items():
        cell_obj = upgrade_legacy_grid_cell(cell_obj)
        if cell_obj is None or cell_obj.output is None:
            continue
        output = cell_obj.output
        config = output.grp_config
        r1 = output.r1
        if r1 is None:
            problems.append((spec.proj_name, "missing r1 relationship"))
            continue
        record = {
            "proj_name": spec.proj_name,
            "project_root": spec.project_root,
            "pair_label": spec.pair_label,
            "row_category": spec.row_category,
            "selection_policy": SELECTION_POLICY,
            "source_object_grid": spec.grid_path,
            "E": primitive(getattr(config, "E", None)),
            "tau": primitive(getattr(config, "tau", None)),
            "Tp": primitive(getattr(config, "Tp", None)),
            "knn": primitive(getattr(config, "knn", None)),
            "col_var_id": primitive(getattr(config, "col_var_id", None)),
            "target_var_id": primitive(getattr(config, "target_var_id", None)),
            "var_x": primitive(getattr(r1, "var_x", getattr(config, "var_x", None))),
            "var_y": primitive(getattr(r1, "var_y", getattr(config, "var_y", None))),
            "relationship_id": primitive(getattr(r1, "r_id", "r1")),
        }
        record.update({column: primitive(getattr(r1, column, None)) for column in METRIC_COLUMNS})
        dyad_records.append(record)

    dyad_df = pd.DataFrame(dyad_records)
    if dyad_df.empty:
        problems.append((spec.proj_name, "no grid records"))
        continue
    duplicate_count = int(dyad_df.duplicated(["E", "tau"]).sum())
    if duplicate_count:
        problems.append((spec.proj_name, f"{duplicate_count} duplicated E/tau rows"))
    missing_metrics = dyad_df[["maxlibsize_rho", "lag", "peak_end",
                               "surr_rx_outperforming_frac", "surr_ry_outperforming_frac"]].isna().any(axis=1)
    if missing_metrics.any():
        # A complete configuration grid can legitimately include cells with
        # no selected CCM result. Preserve nulls so ResultsGrid masks them.
        metric_gaps.append({"proj_name": spec.proj_name,
                            "rows_without_selected_metric": int(missing_metrics.sum())})
    records.extend(dyad_records)

if problems:
    problem_text = "\n".join(f"- {dyad}: {problem}" for dyad, problem in problems)
    raise RuntimeError(f"Export stopped; the precomputed grids are incomplete:\n{problem_text}")

final_result_grid = pd.DataFrame(records).sort_values(
    ["pair_label", "row_category", "proj_name", "tau", "E"]
).reset_index(drop=True)
assert not final_result_grid.duplicated(["proj_name", "selection_policy", "E", "tau"]).any()
display(final_result_grid.head())
print(f"Extracted {len(final_result_grid):,} rows from {final_result_grid.proj_name.nunique()} dyads.")
metric_gap_df = pd.DataFrame(metric_gaps)
if not metric_gap_df.empty:
    print("Rows retained with no selected metric (render as masked cells):")
    display(metric_gap_df)


,proj_name,project_root,pair_label,row_category,selection_policy,source_object_grid,E,tau,Tp,knn,...,maxlibsize_rho,lag,peak_start,peak_end,surr_rx_count,surr_rx_count_outperforming,surr_rx_outperforming_frac,surr_ry_count,surr_ry_count_outperforming,surr_ry_outperforming_frac
0,alley00gisp2multiproxy1kyrfir_vieira11tsianom1...,hol_temp_tsi_ccm1k,Alley-Vieira,1kyrfir,object_grid_precomputed,/Users/jlanders/PycharmProjects/CCM_software/h...,4,1,1,20,...,0.180120,7,2,16,200,3,0.015,200,0,0.0
1,alley00gisp2multiproxy1kyrfir_vieira11tsianom1...,hol_temp_tsi_ccm1k,Alley-Vieira,1kyrfir,object_grid_precomputed,/Users/jlanders/PycharmProjects/CCM_software/h...,5,1,1,20,...,0.177936,7,1,11,200,4,0.020,200,0,0.0
2,alley00gisp2multiproxy1kyrfir_vieira11tsianom1...,hol_temp_tsi_ccm1k,Alley-Vieira,1kyrfir,object_grid_precomputed,/Users/jlanders/PycharmProjects/CCM_software/h...,6,1,1,20,...,0.227721,27,20,32,200,1,0.005,200,0,0.0
3,alley00gisp2multiproxy1kyrfir_vieira11tsianom1...,hol_temp_tsi_ccm1k,Alley-Vieira,1kyrfir,object_grid_precomputed,/Users/jlanders/PycharmProjects/CCM_software/h...,7,1,1,20,...,0.236949,28,20,33,200,0,0.000,200,0,0.0
4,alley00gisp2multiproxy1kyrfir_vieira11tsianom1...,hol_temp_tsi_ccm1k,Alley-Vieira,1kyrfir,object_grid_precomputed,/Users/jlanders/PycharmProjects/CCM_software/h...,8,1,1,20,...,0.239717,29,20,34,200,0,0.000,200,0,0.0


Extracted 1,242 rows from 30 dyads.
Rows retained with no selected metric (render as masked cells):


,proj_name,rows_without_selected_metric
0,Erb22daGMST_Wu18TSI,5
1,NGRIP1Seierstad14d18O_Wu18TSI,40
2,NGRIPMartin24Tanom_Wu18TSI,42


## Validate and write the final artifact

The write happens only after structural validation, so absent or duplicated
grids cannot silently replace a complete previous export.  Grid dimensions
are deliberately not fixed: the existing experiments use different valid
`E`/`tau` ranges (for example 35, 40, 42, or 48 cells).


In [5]:
coverage = final_result_grid.groupby("proj_name").size().rename("rows")
if (coverage == 0).any():
    raise RuntimeError(f"Empty dyad exports:\n{coverage[coverage == 0]}")

output_dir.mkdir(parents=True, exist_ok=True)
final_result_grid.to_parquet(output_path, index=False)

readme_path.write_text(f'''# Final CCM result grid

Generated by `{NOTEBOOK_NAME}` on {datetime.now(timezone.utc).isoformat()}.

This parquet is a plot-ready export of the selected `r1` metrics stored in
upgraded CCM object grids.  It contains one row per `proj_name`, `E`, `tau`,
and `selection_policy`; it does not contain raw observations or CCM output
tables.

`selection_policy={SELECTION_POLICY}` means the export preserves the lag and
surrogate metrics already saved in each upgraded object grid.  It does not recalculate them.

Core plotting columns: `E`, `tau`, `maxlibsize_rho`, `delta_rho`, `lag`,
`peak_start`, `peak_end`, `surr_rx_outperforming_frac`, and
`surr_ry_outperforming_frac`.
''')

print(f"Wrote {len(final_result_grid):,} rows ({output_path.stat().st_size / 1024:.1f} KiB)")
print(f"Wrote {readme_path}")
display(coverage.to_frame())


Wrote 1,466 rows (55.6 KiB)
Wrote /Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm/mixed/tmp/final_result_grid_README.md


,rows
proj_name,
Erb22daGMSTLinear_Vieira11TSILinear,35
Erb22daGMSTLinear_Wu18TSILinear,48
Erb22daGMST_Vieira11TSI,112
Erb22daGMST_Wu18TSI,112
GISP2Alley00TanomLinear_Vieira11TSILinear,35
GISP2Alley00TanomLinear_Wu18TSILinear,48
GISP2Alley00Tanom_Vieira11TSI,112
GISP2Alley00Tanom_Wu18TSI,112
GISP2Doering22T15NLinear_Wu18TSILinear,48


## Use from a figure notebook

```python
results = pd.read_parquet(
    paleobook_dir / "hol_temp_tsi_ccm/mixed/tmp/final_result_grid.parquet"
)
dyad_df = results.query(
    "proj_name == @proj_name and selection_policy == 'object_grid_precomputed'"
)
cell = GridCell(row, col, output=dyad_df)
```

The figure notebook can then build its lightweight `(row, col)` object grid
from these DataFrame-backed `GridCell`s and pass them directly to
`ResultsGrid`.
